In [ ]:
"""
Screening Credit Agentic AI - Synthetic Dataset Generator
============================================================
Generate 7 tabel relasional untuk training model screening kredit retail
banking (5C: Character, Capacity, Collateral, Condition + Capital/Identity).

Semua tabel terhubung lewat NIK (foreign key), KECUALI retail_customer_profile
yang punya application_id sebagai primary key (1 NIK bisa punya banyak
application_id kalau mengajukan berkali-kali, tapi di sini kita generate
1 aplikasi per NIK dulu untuk versi awal).

Cara pakai di Google Colab:
    1. Copy semua isi file ini ke satu cell
    2. Run
    3. 7 file CSV akan tersimpan di /content/dataset/
       (retail_customer_profile.csv, dukcapil.csv, slik_credit_history.csv,
        dhn.csv, agunan_atr_bpn.csv, laporan_keuangan.csv, bank_account.csv)

PENTING saat load ulang CSV-nya nanti (termasuk di tahap join):
    Selalu paksa NIK dibaca sebagai teks, JANGAN biarkan pandas nebak tipenya,
    kalau tidak, 16 digit NIK bisa kepotong presisinya jadi angka:
        pd.read_csv("dukcapil.csv", dtype={"NIK": str})

Catatan penting: nilai tanah/bangunan per kelurahan di sini adalah ESTIMASI
SINTETIS yang dibuat plausible per tingkatan wilayah (bukan data appraisal
resmi/real) - cukup untuk keperluan training model & demo, BUKAN untuk
keputusan bisnis nyata.
"""

import random
import numpy as np
import pandas as pd
from datetime import date, timedelta
import os

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

# Generator acak TERPISAH untuk field tambahan (current_balance, RM assignment)
# supaya penambahan-penambahan ini TIDAK menggeser urutan angka acak yang
# dipakai tabel/kolom lain yang sudah ada sebelumnya.
CB_RNG = np.random.default_rng(SEED + 1)   # khusus current_balance
RM_RNG = np.random.default_rng(SEED + 2)   # khusus penugasan RM

N_CUSTOMERS = 3000          # jumlah nasabah/debitur unik
OUT_DIR = "/content/dataset" if os.path.isdir("/content") else "../data/raw_new"
os.makedirs(OUT_DIR, exist_ok=True)

# =========================================================================
# 0. REFERENCE / LOOKUP DATA
# =========================================================================

FIRST_NAMES_M = ["Budi","Agus","Andi","Rizky","Dedi","Hendra","Yusuf","Fajar",
    "Wahyu","Bambang","Eko","Rudi","Slamet","Joko","Hadi","Ahmad","Dimas",
    "Arif","Taufik","Iwan","Gunawan","Sutrisno","Anton","Rian","Doni",
    "Yudi","Fauzi","Irfan","Bayu","Krisna"]
FIRST_NAMES_F = ["Siti","Dewi","Rina","Ani","Wulan","Sri","Yuni","Fitri",
    "Indah","Lestari","Ratna","Maya","Putri","Ika","Novi","Wati","Ayu",
    "Dian","Rita","Nina","Sari","Yanti","Lina","Desi","Tri","Retno",
    "Kartika","Anggi","Melati","Suryani"]
LAST_NAMES = ["Santoso","Wijaya","Kurniawan","Saputra","Setiawan","Pratama",
    "Hidayat","Nugroho","Firmansyah","Susanto","Gunawan","Halim","Wibowo",
    "Permana","Suryadi","Handoko","Kusuma","Rahman","Siregar","Simanjuntak",
    "Tanjung","Lubis","Hutapea","Panjaitan","Situmorang"]

RELIGIONS = ["ISLAM","KRISTEN","KATOLIK","HINDU","BUDDHA","KONGHUCU"]
MARITAL = ["Menikah","Belum Menikah","Cerai Hidup","Cerai Mati"]
EDUCATION = ["SMA/SMK","D3","S1","S2"]
BLOOD_TYPE = ["A","B","AB","O"]

INDUSTRIES = {
    "Perdagangan": ["Distributor Elektronik","Toko Sembako","Grosir Pakaian",
                    "Distributor Bahan Bangunan","Toko Alat Tulis"],
    "Kuliner": ["Restoran","Katering","Warung Makan","Bakery"],
    "Jasa": ["Bengkel","Laundry","Percetakan","Jasa Konstruksi Kecil"],
    "Manufaktur": ["Konveksi","Furniture","Pengolahan Makanan Ringan"],
    "Pertanian": ["Distributor Hasil Tani","Peternakan Ayam"],
    "Transportasi": ["Ekspedisi Kecil","Rental Kendaraan"],
}
# Bobot risiko sektor untuk "Condition" (dipakai nanti di Risk Agent, bukan
# dipakai untuk generate label secara langsung supaya label tidak terlalu bocor)
INDUSTRY_RISK = {
    "Perdagangan": 0.05, "Kuliner": 0.10, "Jasa": 0.05,
    "Manufaktur": 0.08, "Pertanian": 0.15, "Transportasi": 0.12,
}

PROVINCES_CITIES = {
    "DKI Jakarta": ["Jakarta Selatan","Jakarta Pusat","Jakarta Timur","Jakarta Barat","Jakarta Utara"],
    "Jawa Barat": ["Bekasi","Depok","Bogor","Tangerang Selatan"],
    "Banten": ["Tangerang"],
}
REGIONS = ["Region 1","Region 2","Region 3","Region 4"]
BRANCHES = ["KCP Tebet","KCP Kelapa Gading","KCP Bekasi Barat","KCP Depok Margonda",
    "KCP Bogor Baranangsiang","KCP Tangerang BSD","KCP Cikini","KCP Kemang",
    "KCP Pluit","KCP Cibubur"]

# Dipakai KHUSUS oleh rm_master (tabel baru) - tidak menyentuh kolom "region"
# yang sudah ada di retail_customer_profile (itu tetap independen seperti semula)
BRANCH_TO_REGION = {
    "KCP Tebet": "Region 1", "KCP Cikini": "Region 1", "KCP Kemang": "Region 1",
    "KCP Pluit": "Region 2", "KCP Kelapa Gading": "Region 2", "KCP Cibubur": "Region 2",
    "KCP Bekasi Barat": "Region 3", "KCP Depok Margonda": "Region 3",
    "KCP Bogor Baranangsiang": "Region 4", "KCP Tangerang BSD": "Region 4",
}
RM_PER_BRANCH = 4
RM_LEVELS = ["Junior RB", "Senior RB"]

# Kelurahan lookup untuk agunan (Jabodetabek) -> (provinsi, kota, kecamatan,
# harga tanah/m2 (juta), harga bangunan/m2 (juta)) - ESTIMASI, bukan data resmi
KELURAHAN_LOOKUP = [
    ("DKI Jakarta","Jakarta Selatan","Tebet","Tebet Timur", 28, 5.5),
    ("DKI Jakarta","Jakarta Selatan","Kebayoran Baru","Gunung",45, 6.0),
    ("DKI Jakarta","Jakarta Selatan","Pancoran","Duren Tiga", 30, 5.5),
    ("DKI Jakarta","Jakarta Pusat","Menteng","Menteng", 55, 6.5),
    ("DKI Jakarta","Jakarta Pusat","Cikini","Cikini", 40, 6.0),
    ("DKI Jakarta","Jakarta Timur","Kramat Jati","Kramat Jati", 18, 4.5),
    ("DKI Jakarta","Jakarta Timur","Cakung","Cakung Barat", 12, 4.0),
    ("DKI Jakarta","Jakarta Barat","Kebon Jeruk","Sukabumi Selatan", 22, 5.0),
    ("DKI Jakarta","Jakarta Barat","Cengkareng","Cengkareng Barat", 16, 4.2),
    ("DKI Jakarta","Jakarta Utara","Kelapa Gading","Kelapa Gading Barat", 25, 5.2),
    ("DKI Jakarta","Jakarta Utara","Pluit","Pluit", 27, 5.3),
    ("Jawa Barat","Bekasi","Bekasi Barat","Bintara", 9, 3.8),
    ("Jawa Barat","Bekasi","Bekasi Timur","Margahayu", 8, 3.6),
    ("Jawa Barat","Depok","Beji","Kemiri Muka", 10, 3.8),
    ("Jawa Barat","Depok","Sukmajaya","Mekarjaya", 8.5, 3.6),
    ("Jawa Barat","Bogor","Bogor Tengah","Paledang", 7, 3.4),
    ("Jawa Barat","Tangerang Selatan","Serpong","Rawa Buntu", 12, 4.0),
    ("Banten","Tangerang","Karawaci","Bojong Jaya", 9, 3.6),
    ("Banten","Tangerang","Cipondoh","Poris Plawad", 7.5, 3.4),
]

ASSET_TYPES = ["Tanah","Rumah","Ruko","Gudang"]
CERT_TYPES = ["SHM","HGB"]
LOAN_TYPES = ["KMK","KI","KPR","KKB","KK"]  # Kredit Modal Kerja, Investasi, Pemilikan Rumah, Kendaraan Bermotor, Konsumtif
COLLECT_MAP = {1:"Lancar", 2:"Dalam Perhatian Khusus (DPK)", 3:"Kurang Lancar",
                4:"Diragukan", 5:"Macet"}
OTHER_BANKS = ["Bank Mandiri","Bank BCA","Bank BRI","Bank BNI","Bank CIMB Niaga",
    "Bank Danamon","Bank Permata","Bank OCBC NISP","Bank Panin","BPR Mitra Usaha"]
DHN_REASONS = ["Tunggakan kredit >90 hari di bank lain","Terlibat kasus fraud dokumen",
    "Kredit macet yang belum diselesaikan","Cek/giro kosong berulang",
    "Laporan pihak ketiga terkait sengketa usaha"]

def random_date(start_year, end_year):
    start = date(start_year, 1, 1)
    end = date(end_year, 8, 22)
    delta = (end - start).days
    return start + timedelta(days=random.randint(0, delta))

# kode wilayah (kab/kota) 6-digit ala Kemendagri - plausible, dipakai konsisten dgn kota di dukcapil
KODE_WILAYAH = {
    "Jakarta Selatan": "317401", "Jakarta Pusat": "317101", "Jakarta Timur": "317501",
    "Jakarta Barat": "317301", "Jakarta Utara": "317201",
    "Bekasi": "327501", "Depok": "327601", "Bogor": "327101",
    "Tangerang Selatan": "367401", "Tangerang": "367101",
}

def gen_nik(kota, tanggal_lahir, gender, idx):
    # NIK 16 digit sesuai standar: kode_wilayah(6) + ddmmyy(6, +40 hari utk perempuan) + urutan(4)
    wilayah = KODE_WILAYAH.get(kota, "310101")
    d = tanggal_lahir.day + (40 if gender == "Perempuan" else 0)
    m, y = tanggal_lahir.month, tanggal_lahir.year % 100
    return f"{wilayah}{d:02d}{m:02d}{y:02d}{idx:04d}"


# =========================================================================
# 1. DUKCAPIL (identitas dasar, mengikuti field KTP)
# =========================================================================
def generate_dukcapil(n):
    rows = []
    for i in range(1, n+1):
        gender = random.choice(["Laki-Laki","Perempuan"])
        fname = random.choice(FIRST_NAMES_M if gender=="Laki-Laki" else FIRST_NAMES_F)
        lname = random.choice(LAST_NAMES)
        nama = f"{fname} {lname}"
        prov = random.choice(list(PROVINCES_CITIES.keys()))
        kota = random.choice(PROVINCES_CITIES[prov])
        tgl_lahir = random_date(1965, 2003)
        nik = gen_nik(kota, tgl_lahir, gender, i)
        rows.append({
            "dukcapil_id": f"DKC{i:06d}",
            "NIK": nik,
            "nama": nama,
            "tempat_lahir": kota,
            "tanggal_lahir": tgl_lahir.isoformat(),
            "jenis_kelamin": gender,
            "golongan_darah": random.choice(BLOOD_TYPE),
            "alamat": f"Jl. {random.choice(LAST_NAMES)} No. {random.randint(1,150)}",
            "rt_rw": f"{random.randint(1,12):03d}/{random.randint(1,10):03d}",
            "kelurahan_desa": random.choice(["Sukamaju","Sukajadi","Cempaka Putih",
                "Kebon Baru","Duren Sawit","Rawa Bunga","Cipete","Bintaro"]),
            "kecamatan": random.choice(["Tebet","Kramat Jati","Cengkareng",
                "Bekasi Timur","Sukmajaya","Serpong"]),
            "kota_kabupaten": kota,
            "provinsi": prov,
            "agama": random.choice(RELIGIONS),
            "status_perkawinan": random.choice(MARITAL),
            "pekerjaan": "Wiraswasta",
            "kewarganegaraan": "WNI",
            "berlaku_hingga": "SEUMUR HIDUP",
        })
    return pd.DataFrame(rows)


# =========================================================================
# 2. AGUNAN / ATR-BPN
# =========================================================================
def generate_agunan(dukcapil_df):
    rows = []
    agunan_lookup = {}  # NIK -> summary dict (dipakai utk customer_profile)
    for i, r in enumerate(dukcapil_df.itertuples(), start=1):
        nik = r.NIK
        prov, kota, kec, kel, harga_tanah, harga_bangunan = random.choice(KELURAHAN_LOOKUP)
        asset_type = random.choices(ASSET_TYPES, weights=[0.25,0.30,0.35,0.10])[0]
        land_area = round(np.random.uniform(60, 400), 1)
        building_area = 0.0 if asset_type == "Tanah" else round(land_area * np.random.uniform(0.5, 1.3), 1)
        # variasi harga per unit +/- 15%
        htn = round(harga_tanah * np.random.uniform(0.85, 1.15), 2)
        hbg = round(harga_bangunan * np.random.uniform(0.85, 1.15), 2)
        nilai_tanah = round(land_area * htn * 1_000_000)
        nilai_bangunan = round(building_area * hbg * 1_000_000)
        total_value = nilai_tanah + nilai_bangunan
        ownership_match = np.random.choice(["Ya","Tidak"], p=[0.94, 0.06])
        row = {
            "atr_bpn_id": f"ATR{i:06d}",
            "NIK": nik,
            "asset_type": asset_type,
            "certificate_type": random.choice(CERT_TYPES),
            "certificate_number": f"{random.randint(10000,99999)}/{kel}",
            "provinsi": prov, "kota": kota, "kecamatan": kec, "kelurahan": kel,
            "land_area_m2": land_area,
            "building_area_m2": building_area,
            "nilai_tanah_per_m2": int(htn * 1_000_000),
            "nilai_bangunan_per_m2": int(hbg * 1_000_000),
            "nilai_tanah_total": nilai_tanah,
            "nilai_bangunan_total": nilai_bangunan,
            "total_collateral_value": total_value,
            "ownership_match": ownership_match,
        }
        rows.append(row)
        agunan_lookup[nik] = row
    return pd.DataFrame(rows), agunan_lookup


# =========================================================================
# 3. SLIK CREDIT HISTORY (1-3 fasilitas kredit per NIK, bisa juga 0)
# =========================================================================
def generate_slik(dukcapil_df):
    rows = []
    slik_summary = {}  # NIK -> {"worst_collect":.., "total_installment":.., "n_loans":..}
    rid = 1
    for r in dukcapil_df.itertuples():
        nik = r.NIK
        n_loans = np.random.choice([0,1,2,3], p=[0.15,0.40,0.30,0.15])
        worst = 1
        total_installment = 0
        for _ in range(n_loans):
            plafond = int(np.random.choice([25,50,75,100,150,200,300,500]) * 1_000_000)
            outstanding = int(plafond * np.random.uniform(0.2, 0.95))
            tenor = int(np.random.choice([12,24,36,48,60]))
            installment = int(plafond / tenor * np.random.uniform(1.02,1.15))
            # sebagian besar Lancar, sebagian kecil bermasalah (realistis)
            collect = np.random.choice([1,2,3,4,5], p=[0.72,0.14,0.07,0.04,0.03])
            worst = max(worst, collect)
            total_installment += installment
            rows.append({
                "slik_record_id": f"SLK{rid:06d}",
                "NIK": nik,
                "inquiry_date": random_date(2024,2026).isoformat(),
                "bank_name": random.choice(OTHER_BANKS),
                "loan_type": random.choice(LOAN_TYPES),
                "plafond": plafond,
                "outstanding_balance": outstanding,
                "installment_amount": installment,
                "tenor_month": tenor,
                "collectability": int(collect),
                "collectability_label": COLLECT_MAP[collect],
            })
            rid += 1
        slik_summary[nik] = {"worst_collect": worst, "total_installment": total_installment, "n_loans": n_loans}
    return pd.DataFrame(rows), slik_summary


# =========================================================================
# 4. DHN (Daftar Hitam Nasional) - korelasi dgn worst SLIK collectability
# =========================================================================
def generate_dhn(dukcapil_df, slik_summary):
    rows = []
    dhn_lookup = {}
    for i, r in enumerate(dukcapil_df.itertuples(), start=1):
        nik = r.NIK
        worst = slik_summary[nik]["worst_collect"]
        # makin buruk kolektibilitas SLIK, makin besar peluang masuk DHN
        p_blacklist = {1:0.01, 2:0.03, 3:0.10, 4:0.25, 5:0.45}[worst]
        status = np.random.choice(["Ya","Tidak"], p=[p_blacklist, 1-p_blacklist])
        reason = random.choice(DHN_REASONS) if status == "Ya" else ""
        row = {
            "dhn_id": f"DHN{i:06d}",
            "NIK": nik,
            "status_dhn": status,
            "alasan": reason,
            "tanggal_input": random_date(2023,2026).isoformat(),
        }
        rows.append(row)
        dhn_lookup[nik] = status
    return pd.DataFrame(rows), dhn_lookup


# =========================================================================
# 5. LAPORAN KEUANGAN (2 tahun: 2024 & 2025)
# =========================================================================
def generate_laporan_keuangan(dukcapil_df):
    rows = []
    fin_summary = {}
    rid = 1
    for r in dukcapil_df.itertuples():
        nik = r.NIK
        revenue_2024 = np.random.lognormal(mean=16.8, sigma=0.6)  # ~ puluhan-ratusan juta s/d miliaran
        growth = np.random.normal(0.12, 0.20)  # rata2 tumbuh 12%, bisa negatif
        revenue_2025 = revenue_2024 * (1 + growth)
        margin = np.clip(np.random.normal(0.11, 0.05), 0.01, 0.35)
        recs = []
        for yr, rev in [(2024, revenue_2024), (2025, revenue_2025)]:
            net_profit = rev * margin * np.random.uniform(0.85,1.15)
            total_asset = rev * np.random.uniform(1.1, 2.0)
            total_liability = total_asset * np.random.uniform(0.2, 0.7)
            op_cf = net_profit * np.random.uniform(0.8, 1.4)
            row = {
                "laporan_id": f"FIN{rid:06d}", "NIK": nik, "year": yr,
                "revenue": int(rev), "net_profit": int(net_profit),
                "total_asset": int(total_asset), "total_liability": int(total_liability),
                "operating_cashflow": int(op_cf),
            }
            rows.append(row); recs.append(row); rid += 1
        fin_summary[nik] = {
            "revenue_growth": growth,
            "latest_revenue": recs[1]["revenue"],
            "latest_net_profit": recs[1]["net_profit"],
            "latest_liability": recs[1]["total_liability"],
        }
    return pd.DataFrame(rows), fin_summary


# =========================================================================
# 6. BANK ACCOUNT (1-2 rekening per NIK, korelasi dgn omset)
# =========================================================================
def generate_bank_account(dukcapil_df, fin_summary):
    rows = []
    cf_summary = {}
    aid = 1
    for r in dukcapil_df.itertuples():
        nik = r.NIK
        n_acc = np.random.choice([1,2], p=[0.65,0.35])
        monthly_rev = fin_summary[nik]["latest_revenue"] / 12
        best_avg_balance = 0
        for _ in range(n_acc):
            avg_credit = monthly_rev * np.random.uniform(0.6, 1.1)
            avg_debit = avg_credit * np.random.uniform(0.7, 0.98)
            avg_balance = max(avg_credit - avg_debit, 0) * np.random.uniform(40, 100)
            best_avg_balance = max(best_avg_balance, avg_balance)

            # --- SAMA PERSIS seperti sebelumnya: urutan & isi field, dan
            # urutan pemanggilan np.random di sini, TIDAK diubah sama sekali ---
            account = {
                "account_id": f"ACC{aid:06d}",
                "NIK": nik,
                "account_number": f"{random.randint(1000000000,9999999999):010d}",
                "bank_name": random.choice(["BNI"] + OTHER_BANKS),
                "account_type": random.choice(["Giro","Tabungan"]),
                "account_status": np.random.choice(["Aktif","Dormant"], p=[0.93,0.07]),
                "opened_date": random_date(2015,2025).isoformat(),
                "average_balance_6m": int(avg_balance),
                "average_monthly_credit": int(avg_credit),
                "average_monthly_debit": int(avg_debit),
                "transaction_frequency_monthly": int(np.random.uniform(20,200)),
                "overdraft_count_6m": int(np.random.choice([0,0,0,1,2,3], p=[0.6,0.15,0.1,0.08,0.04,0.03])),
            }

            # --- BARU: current_balance ditambahkan SETELAH dict di atas
            # lengkap, pakai CB_RNG (generator terpisah) - tidak menyentuh
            # np.random global sama sekali, jadi tidak mengubah urutan
            # angka acak untuk tabel lain maupun field bank_account lainnya.
            if account["account_status"] == "Dormant":
                current_balance = avg_balance * CB_RNG.uniform(0.05, 0.25)
            elif account["overdraft_count_6m"] > 0 and CB_RNG.random() < 0.12:
                current_balance = -avg_debit * CB_RNG.uniform(0.02, 0.15)
            else:
                current_balance = avg_balance * CB_RNG.uniform(0.4, 1.8)
            account["current_balance"] = int(current_balance)

            rows.append(account)
            aid += 1
        cf_summary[nik] = {"best_avg_balance": best_avg_balance}
    return pd.DataFrame(rows), cf_summary


# =========================================================================
# 6b. RM MASTER (data Relationship Banking Officer) - TABEL BARU
# =========================================================================
def generate_rm_master():
    rows = []
    rm_lookup = {branch: [] for branch in BRANCHES}  # branch_name -> list of rm_id
    rid = 1
    for branch in BRANCHES:
        for _ in range(RM_PER_BRANCH):
            gender = RM_RNG.choice(["Laki-Laki", "Perempuan"])
            fname = RM_RNG.choice(FIRST_NAMES_M if gender == "Laki-Laki" else FIRST_NAMES_F)
            lname = RM_RNG.choice(LAST_NAMES)
            rm_id = f"RM{rid:04d}"
            rows.append({
                "rm_id": rm_id,
                "rm_name": f"{fname} {lname}",
                "branch_name": branch,
                "region": BRANCH_TO_REGION[branch],
                "jabatan": "Relationship Banking Officer",
                "level": RM_RNG.choice(RM_LEVELS, p=[0.6, 0.4]),
                "join_date": (date(2015, 1, 1) + timedelta(
                    days=int(RM_RNG.integers(0, (date(2025, 12, 31) - date(2015, 1, 1)).days)))).isoformat(),
            })
            rm_lookup[branch].append(rm_id)
            rid += 1
    return pd.DataFrame(rows), rm_lookup


# =========================================================================
# 7. RETAIL CUSTOMER PROFILE (application) + LABEL diterima/ditolak
# =========================================================================
def compute_label_score(worst_collect, dhn_status, growth, net_profit, dsr,
                          collateral_ratio, industry):
    # semua sub-skor di-normalisasi 0..1, makin tinggi makin baik
    s_character = {1:1.0, 2:0.8, 3:0.5, 4:0.25, 5:0.0}[worst_collect]
    if dhn_status == "Ya":
        s_character = min(s_character, 0.1)
    s_capacity = np.clip(0.5 + growth*1.2, 0, 1) * 0.5 + np.clip(1 - dsr, 0, 1) * 0.5
    s_capacity = np.clip(s_capacity, 0, 1)
    s_collateral = np.clip(collateral_ratio / 1.5, 0, 1)  # LTV>=150% -> skor penuh
    s_condition = 1 - INDUSTRY_RISK.get(industry, 0.1) * 4  # sektor riskier -> skor turun sedikit
    s_condition = np.clip(s_condition, 0, 1)

    # bobot 5C (Character & Capacity paling berat, sesuai praktik umum)
    score = 0.35*s_character + 0.30*s_capacity + 0.20*s_collateral + 0.15*s_condition
    score += np.random.normal(0, 0.05)  # noise supaya tidak terlalu deterministik
    return np.clip(score, 0, 1)

def generate_customer_profile(dukcapil_df, agunan_lookup, slik_summary, dhn_lookup,
                                fin_summary, cf_summary, rm_lookup):
    rows = []
    for i, r in enumerate(dukcapil_df.itertuples(), start=1):
        nik = r.NIK
        prov, kota = r.provinsi, r.kota_kabupaten
        legal_entity = random.choice(["PT","CV","UD"])
        industry = random.choice(list(INDUSTRIES.keys()))
        sub_industry = random.choice(INDUSTRIES[industry])
        business_age = int(np.random.uniform(1, 20))
        employee_count = int(np.random.uniform(2, 80))
        monthly_turnover = fin_summary[nik]["latest_revenue"] / 12

        agunan = agunan_lookup[nik]
        loan_requested = int(np.random.choice([50,75,100,150,200,300,500,750,1000]) * 1_000_000)
        loan_requested = min(loan_requested, 10_000_000_000)
        collateral_ratio = round(agunan["total_collateral_value"] / max(loan_requested,1), 2)
        collateral_size_m2 = round(agunan["land_area_m2"] + agunan["building_area_m2"], 1)

        slik = slik_summary[nik]
        # estimasi cicilan baru dari pengajuan ini (asumsi tenor 36 bulan, bunga flat ~12%/th)
        new_installment = loan_requested/36 * 1.12
        dsr = (slik["total_installment"] + new_installment) / max(monthly_turnover, 1)

        score = compute_label_score(
            worst_collect=slik["worst_collect"], dhn_status=dhn_lookup[nik],
            growth=fin_summary[nik]["revenue_growth"], net_profit=fin_summary[nik]["latest_net_profit"],
            dsr=dsr, collateral_ratio=collateral_ratio, industry=industry,
        )
        label = "Diterima" if score >= 0.55 else "Ditolak"

        row = {
            "application_id": f"APP{2026}{i:05d}",
            "NIK": nik,
            "cif_number": f"CIF{1000000+i}",
            "application_date": random_date(2025,2026).isoformat(),
            "customer_type": "UMKM",
            "company_name": f"{random.choice(['PT','CV','UD'])} {random.choice(LAST_NAMES)} {random.choice(['Jaya','Makmur','Sejahtera','Abadi','Mandiri'])}",
            "legal_entity": legal_entity,
            "owner_name": r.nama,
            "owner_gender": "L" if r.jenis_kelamin=="Laki-Laki" else "P",
            "owner_age": date.today().year - int(r.tanggal_lahir[:4]),
            "owner_marital_status": r.status_perkawinan,
            "owner_education": random.choice(EDUCATION),
            "province": prov, "city": kota,
            "district": r.kecamatan, "region": random.choice(REGIONS),
            "branch_name": random.choice(BRANCHES),
            "industry": industry, "sub_industry": sub_industry,
            "business_age_year": business_age,
            "employee_count": employee_count,
            "monthly_turnover_est": int(monthly_turnover),
            "transaction_frequency_monthly": int(np.random.uniform(30,200)),
            "loan_requested": loan_requested,
            "collateral_type": agunan["asset_type"],
            "collateral_location": f"{agunan['kelurahan']}, {agunan['kota']}",
            "collateral_province": agunan["provinsi"], "collateral_city": agunan["kota"],
            "collateral_size_m2": collateral_size_m2,
            "collateral_market_value": agunan["total_collateral_value"],
            "collateral_liquidation_value": int(agunan["total_collateral_value"] * 0.8),
            "collateral_ratio": collateral_ratio,
            "certificate_type": agunan["certificate_type"],
            "ownership_match": agunan["ownership_match"],
            "estimated_dsr": round(min(dsr,3.0), 2),
            "eligibility_score": round(float(score), 3),
            "label": label,
        }

        # --- BARU: penugasan RM, ditambahkan SETELAH dict di atas lengkap ---
        # Pakai RM_RNG (generator terpisah) & branch_name yang SUDAH ada di
        # atas (tidak diubah) - jadi tidak menyentuh np.random/random module
        # global sama sekali. RM dipilih dari RM yang bertugas di cabang yang
        # sama dengan branch_name pengajuan ini (realistis: RM 1 cabang
        # menangani nasabah cabang itu juga).
        row["rm_id"] = RM_RNG.choice(rm_lookup[row["branch_name"]])

        rows.append(row)
    return pd.DataFrame(rows)


# =========================================================================
# MAIN
# =========================================================================
def main():
    print(f"Generating {N_CUSTOMERS} customers...")
    dukcapil_df = generate_dukcapil(N_CUSTOMERS)
    agunan_df, agunan_lookup = generate_agunan(dukcapil_df)
    slik_df, slik_summary = generate_slik(dukcapil_df)
    dhn_df, dhn_lookup = generate_dhn(dukcapil_df, slik_summary)
    fin_df, fin_summary = generate_laporan_keuangan(dukcapil_df)
    bank_df, cf_summary = generate_bank_account(dukcapil_df, fin_summary)
    rm_df, rm_lookup = generate_rm_master()
    profile_df = generate_customer_profile(dukcapil_df, agunan_lookup, slik_summary,
                                            dhn_lookup, fin_summary, cf_summary, rm_lookup)

    tables = {
        "retail_customer_profile": profile_df,
        "dukcapil": dukcapil_df,
        "slik_credit_history": slik_df,
        "dhn": dhn_df,
        "agunan_atr_bpn": agunan_df,
        "laporan_keuangan": fin_df,
        "bank_account": bank_df,
        "rm_master": rm_df,
    }
    for name, df in tables.items():
        if "NIK" in df.columns:
            df["NIK"] = df["NIK"].astype(str)   # cegah NIK kebaca sbg int/float & kepotong presisinya
        if "account_number" in df.columns:
            df["account_number"] = df["account_number"].astype(str)  # sama alasannya (bukan angka matematis)
        path = os.path.join(OUT_DIR, f"{name}.csv")
        df.to_csv(path, index=False)
        print(f"  {name:28s} -> {len(df):6d} baris -> {path}")

    print("\nDistribusi label (retail_customer_profile):")
    print(profile_df["label"].value_counts(normalize=True).round(3))
    print("\nSelesai. Semua file CSV ada di:", OUT_DIR)
    return tables

if __name__ == "__main__":
    tables = main()

Generating 3000 customers...
  retail_customer_profile      ->   3000 baris -> ../data/raw\retail_customer_profile.csv
  dukcapil                     ->   3000 baris -> ../data/raw\dukcapil.csv
  slik_credit_history          ->   4416 baris -> ../data/raw\slik_credit_history.csv
  dhn                          ->   3000 baris -> ../data/raw\dhn.csv
  agunan_atr_bpn               ->   3000 baris -> ../data/raw\agunan_atr_bpn.csv
  laporan_keuangan             ->   6000 baris -> ../data/raw\laporan_keuangan.csv
  bank_account                 ->   4052 baris -> ../data/raw\bank_account.csv
  rm_master                    ->     40 baris -> ../data/raw\rm_master.csv

Distribusi label (retail_customer_profile):
label
Diterima    0.848
Ditolak     0.152
Name: proportion, dtype: float64

Selesai. Semua file CSV ada di: ../data/raw


In [2]:
pd.set_option('display.max_columns', None)

for name, df in tables.items():
    print(f"\n--- Columns for table: {name} ---")
    display(df.head())



--- Columns for table: retail_customer_profile ---


,application_id,NIK,cif_number,application_date,customer_type,company_name,legal_entity,owner_name,owner_gender,owner_age,owner_marital_status,owner_education,province,city,district,region,branch_name,industry,sub_industry,business_age_year,employee_count,monthly_turnover_est,transaction_frequency_monthly,loan_requested,collateral_type,collateral_location,collateral_province,collateral_city,collateral_size_m2,collateral_market_value,collateral_liquidation_value,collateral_ratio,certificate_type,ownership_match,estimated_dsr,eligibility_score,label,rm_id
0,APP202600001,3276010601750001,CIF1000001,2025-07-08,UMKM,UD Santoso Abadi,UD,Budi Panjaitan,L,51,Menikah,S2,Jawa Barat,Depok,Sukmajaya,Region 2,KCP Bogor Baranangsiang,Manufaktur,Konveksi,14,9,2672137,66,300000000,Rumah,"Poris Plawad, Tangerang",Banten,Tangerang,423.4,2328496000,1862796800,7.76,HGB,Ya,3.0,0.804,Diterima,RM0020
1,APP202600002,3172010301920002,CIF1000002,2025-09-01,UMKM,UD Wijaya Mandiri,UD,Andi Hidayat,L,34,Cerai Hidup,S2,DKI Jakarta,Jakarta Utara,Kramat Jati,Region 1,KCP Bekasi Barat,Jasa,Bengkel,1,3,1807917,39,75000000,Rumah,"Pluit, Jakarta Utara",DKI Jakarta,Jakarta Utara,174.8,3724038000,2979230400,49.65,HGB,Ya,3.0,0.683,Diterima,RM0010
2,APP202600003,3671010604800003,CIF1000003,2025-10-27,UMKM,UD Kusuma Sejahtera,CV,Doni Pratama,L,46,Cerai Hidup,S2,Banten,Tangerang,Bekasi Timur,Region 3,KCP Cibubur,Transportasi,Ekspedisi Kecil,17,33,1698736,59,150000000,Tanah,"Sukabumi Selatan, Jakarta Barat",DKI Jakarta,Jakarta Barat,67.0,1681700000,1345360000,11.21,HGB,Ya,3.0,0.435,Ditolak,RM0039
3,APP202600004,3173016001890004,CIF1000004,2025-10-23,UMKM,UD Susanto Makmur,CV,Nina Firmansyah,P,37,Menikah,S2,DKI Jakarta,Jakarta Barat,Sukmajaya,Region 3,KCP Bogor Baranangsiang,Perdagangan,Toko Alat Tulis,12,79,1135105,98,200000000,Ruko,"Tebet Timur, Jakarta Selatan",DKI Jakarta,Jakarta Selatan,200.6,3647200000,2917760000,18.24,HGB,Ya,3.0,0.683,Diterima,RM0017
4,APP202600005,3275011505030005,CIF1000005,2025-08-17,UMKM,CV Wijaya Sejahtera,UD,Sutrisno Nugroho,L,23,Cerai Hidup,S2,Jawa Barat,Bekasi,Kramat Jati,Region 3,KCP Bogor Baranangsiang,Transportasi,Rental Kendaraan,11,5,862772,120,750000000,Ruko,"Kemiri Muka, Depok",Jawa Barat,Depok,316.3,1978268000,1582614400,2.64,HGB,Ya,3.0,0.664,Diterima,RM0019



--- Columns for table: dukcapil ---


,dukcapil_id,NIK,nama,tempat_lahir,tanggal_lahir,jenis_kelamin,golongan_darah,alamat,rt_rw,kelurahan_desa,kecamatan,kota_kabupaten,provinsi,agama,status_perkawinan,pekerjaan,kewarganegaraan,berlaku_hingga
0,DKC000001,3276010601750001,Budi Panjaitan,Depok,1975-01-06,Laki-Laki,B,Jl. Panjaitan No. 27,011/009,Sukajadi,Sukmajaya,Depok,Jawa Barat,HINDU,Menikah,Wiraswasta,WNI,SEUMUR HIDUP
1,DKC000002,3172010301920002,Andi Hidayat,Jakarta Utara,1992-01-03,Laki-Laki,A,Jl. Rahman No. 51,012/009,Cipete,Kramat Jati,Jakarta Utara,DKI Jakarta,HINDU,Cerai Hidup,Wiraswasta,WNI,SEUMUR HIDUP
2,DKC000003,3671010604800003,Doni Pratama,Tangerang,1980-04-06,Laki-Laki,AB,Jl. Setiawan No. 56,006/002,Sukajadi,Bekasi Timur,Tangerang,Banten,ISLAM,Cerai Hidup,Wiraswasta,WNI,SEUMUR HIDUP
3,DKC000004,3173016001890004,Nina Firmansyah,Jakarta Barat,1989-01-20,Perempuan,A,Jl. Wibowo No. 21,009/005,Rawa Bunga,Sukmajaya,Jakarta Barat,DKI Jakarta,KRISTEN,Menikah,Wiraswasta,WNI,SEUMUR HIDUP
4,DKC000005,3275011505030005,Sutrisno Nugroho,Bekasi,2003-05-15,Laki-Laki,B,Jl. Saputra No. 98,005/008,Rawa Bunga,Kramat Jati,Bekasi,Jawa Barat,KATOLIK,Cerai Hidup,Wiraswasta,WNI,SEUMUR HIDUP



--- Columns for table: slik_credit_history ---


...

,laporan_id,NIK,year,revenue,net_profit,total_asset,total_liability,operating_cashflow
0,FIN000001,3276010601750001,2024,29666491,3405584,48148218,27015813,3039455
1,FIN000002,3276010601750001,2025,32065651,3318494,37084511,18787312,4499938
2,FIN000003,3172010301920002,2024,22347316,2907631,27256315,17484128,3572130
3,FIN000004,3172010301920002,2025,21695009,2602253,35323008,9380894,2863796
4,FIN000005,3671010604800003,2024,23914503,5615079,34076107,17322819,7404291



--- Columns for table: bank_account ---


,account_id,NIK,account_number,bank_name,account_type,account_status,opened_date,average_balance_6m,average_monthly_credit,average_monthly_debit,transaction_frequency_monthly,overdraft_count_6m,current_balance
0,ACC000001,3276010601750001,6956650690,Bank BCA,Giro,Aktif,2023-08-14,620254,2196060,2092204,148,0,814529
1,ACC000002,3276010601750001,6225587025,BNI,Giro,Aktif,2020-05-20,1002986,2212722,1737140,76,0,462663
2,ACC000003,3172010301920002,4729122289,Bank BNI,Tabungan,Aktif,2025-03-21,211129,1124770,1040039,122,0,90372
3,ACC000004,3671010604800003,4611355005,Bank BCA,Giro,Dormant,2021-10-23,851330,1394768,1222864,22,1,185456
4,ACC000005,3173016001890004,8727347064,Bank CIMB Niaga,Giro,Dormant,2025-08-11,99532,1172430,1132201,157,0,16664



--- Columns for table: rm_master ---


,rm_id,rm_name,branch_name,region,jabatan,level,join_date
0,RM0001,Ani Tanjung,KCP Tebet,Region 1,Relationship Banking Officer,Junior RB,2017-11-02
1,RM0002,Krisna Permana,KCP Tebet,Region 1,Relationship Banking Officer,Senior RB,2016-10-14
2,RM0003,Dedi Kurniawan,KCP Tebet,Region 1,Relationship Banking Officer,Senior RB,2018-09-19
3,RM0004,Rita Halim,KCP Tebet,Region 1,Relationship Banking Officer,Junior RB,2025-07-02
4,RM0005,Bayu Situmorang,KCP Kelapa Gading,Region 2,Relationship Banking Officer,Senior RB,2025-03-11
